## Step 2+3 - Cut keyframes -> webp (gộp 1 bước)

Gộp `02_cut-frame-video` và `03_keyframe-extra-reduced`: đọc shot boundaries của
Step 1, cắt frame đại diện (first / middle / last mỗi shot) rồi **ghi thẳng
`.webp` đã resize** - không còn bước jpg full-res trung gian.

```
Keyframes/<batch>/<video_name>/<idx:06d>.webp   e.g. Keyframes/L21/L21_V001/000080.webp
frames_meta.json = { "<video_name>_<idx:06d>": {video_name, frame_name, timestamp_ms, fps} }
```

`video_name = <batch>_<video_id>` (vd `L21_V001`); `frame_id = <video_name>_<idx>`
(vd `L21_V001_000080`) là khóa join dùng cho FAISS id_map và các file OCR.

**Vì sao gộp:** Step 2 cũ ghi jpg full-res rồi Step 3 mới thu nhỏ sang webp, nên
lúc chạy phải chứa đồng thời cả hai bản. Với ~500 video đó là ~40-50 GB và làm
`/kaggle/working` (giới hạn ~20 GB) đầy đĩa giữa chừng
(`OSError: [Errno 28] No space left on device`). Ghi thẳng webp giảm ~10 lần dung lượng.

**Khác biệt so với bản cũ (ngoài chuyện gộp):**

* **Resume được** - `frames_done.json` ghi các video đã xong; chạy lại sẽ bỏ qua,
  không phải decode lại từ đầu.
* **Metadata ghi dần** (mỗi `FLUSH_EVERY` video, ghi atomic) thay vì chỉ ghi ở
  cuối vòng lặp - crash giữa chừng không còn mất sạch metadata.
* **Chết to, chết sớm khi hết đĩa** - kiểm tra dung lượng trống trước mỗi video và
  dừng với thông báo rõ ràng, thay vì để `cv2.imwrite` trả `False` âm thầm bỏ frame.
* **`ONLY_BATCHES`** - chạy từng phần batch cho vừa hạn mức đĩa của Kaggle.

> **Khử trùng lặp (cosine):** `DEDUP_THRESHOLD` (mặc định `0.95`) bỏ frame gần-giống
> ngay khi cắt - so cosine trên pixel (ảnh xám 32x32, trừ trung bình) với frame giữ
> gần nhất *trong cùng video*. Đặt `0` để tắt.

In [ ]:
import os
os.environ["OPENCV_FFMPEG_LOGLEVEL"] = "-8"   # tắt cảnh báo decoder H.264 rất ồn
import cv2
import json
import math
import shutil
from pathlib import Path
from PIL import Image
from tqdm import tqdm

In [ ]:
# Tự nhận môi trường: Kaggle đọc từ /kaggle/input, local đọc thẳng trong repo.
IS_KAGGLE = os.path.isdir('/kaggle/working')

if IS_KAGGLE:
    VIDEOS_DIR     = '/kaggle/input/aic-videos'                 # Videos_<batch>/video/*.mp4
    SCENE_JSON_DIR = '/kaggle/input/aic-scenes/SceneJSON'       # <- đổi cho khớp tên dataset output Step 1
    OUT_ROOT       = '/kaggle/working'
else:
    VIDEOS_DIR     = '../../backend/media/videos'
    SCENE_JSON_DIR = '../../backend/media/scenes/SceneJSON'
    OUT_ROOT       = '../../backend/media/frames'

SAVE_DIR  = f'{OUT_ROOT}/Keyframes'
META_PATH = f'{OUT_ROOT}/frames_meta.json'
DONE_PATH = f'{OUT_ROOT}/frames_done.json'    # danh sách video đã cắt xong -> để resume

FINAL_EXT    = 'webp'   # định dạng ghi ra, đồng thời là đuôi ghi trong frames_meta
REDUCE_RATIO = 0.5      # thu nhỏ cạnh (giống Step 3 cũ)
WEBP_QUALITY = 70

# Khử trùng lặp gần-giống bằng cosine trên pixel frame (chưa có embedding ở bước này).
# So mỗi frame ứng viên với frame giữ gần nhất TRONG CÙNG video; > ngưỡng -> bỏ.
# 0 = tắt. 0.90-0.95 hợp lý; cao hơn = bỏ ít hơn. Chỉnh tùy độ 'tĩnh' của cảnh.
DEDUP_THRESHOLD = 0.95

# Chỉ chạy vài batch cho vừa đĩa, vd ['L21', 'L22']. [] = chạy tất cả.
ONLY_BATCHES = []

MIN_FREE_GB = 1.0   # còn ít hơn ngần này thì dừng có kiểm soát (Kaggle /kaggle/working ~20 GB)
FLUSH_EVERY = 10    # ghi frames_meta / frames_done sau mỗi N video

os.makedirs(SAVE_DIR, exist_ok=True)
print('kaggle' if IS_KAGGLE else 'local', '->', SAVE_DIR)

In [ ]:
# Quét đệ quy MỌI file video, không phụ thuộc cách chia thư mục
# (Videos_L21/video/..., Videos_L25_a/..., Videos_L25_b/..., lồng sâu tùy ý).
# video_name = tên file (stem), ví dụ L25_V001.mp4 -> video_name 'L25_V001',
# batch 'L25', video_id 'V001'. Khớp key join của các bước sau.
#
# GỘP THEO TÊN FILE, KHÔNG THEO FOLDER: các folder Videos_L25_a / _a1 / _b đều
# gộp về batch 'L25' vì key lấy từ TÊN FILE. Vài dataset upload LẶP một part
# (vd Videos_L25_a và Videos_L25_a1 chứa cùng dải L25_V001..V088). Ta phân biệt:
#   * cùng tên + CÙNG dung lượng  -> bản sao dư thừa, bỏ qua (giữ bản gặp đầu).
#   * cùng tên + KHÁC dung lượng  -> hai video khác nội dung xài chung id -> DỪNG,
#     vì gộp về cùng batch sẽ mất data (cần đưa hậu tố folder vào khóa).
VIDEO_EXTS = ('.mp4', '.mkv', '.webm', '.avi', '.mov')

all_video_paths = {}
seen = {}                 # video_name -> (path, size) của bản gặp đầu tiên
conflicts = []            # trùng tên NHƯNG khác nội dung (khác size)
redundant = 0             # trùng tên VÀ cùng size -> bản sao, bỏ qua an toàn
skipped = 0
for vid in sorted(Path(VIDEOS_DIR).rglob('*')):
    if not (vid.is_file() and vid.suffix.lower() in VIDEO_EXTS):
        continue
    video_name = vid.stem                              # 'L25_V001'
    if '_' not in video_name:                          # tên không có batch -> bỏ
        skipped += 1
        continue
    if video_name in seen:
        prev_path, prev_size = seen[video_name]
        if vid.stat().st_size == prev_size:            # cùng tên + cùng size -> bản sao
            redundant += 1
        else:                                          # cùng tên, khác nội dung -> xung đột
            conflicts.append((video_name, prev_path, str(vid)))
        continue
    seen[video_name] = (str(vid), vid.stat().st_size)
    batch, video_id = video_name.split('_', 1)         # 'L25', 'V001'
    all_video_paths.setdefault(batch, {})[video_id] = str(vid)

if ONLY_BATCHES:
    missing = [b for b in ONLY_BATCHES if b not in all_video_paths]
    if missing:
        print(f'[WARN] ONLY_BATCHES có batch không tồn tại: {missing}')
    all_video_paths = {b: v for b, v in all_video_paths.items() if b in ONLY_BATCHES}

print('batches:', {k: len(v) for k, v in all_video_paths.items()})
print('tổng video:', sum(len(v) for v in all_video_paths.values()))
if skipped:
    print(f'[WARN] bỏ qua {skipped} file không có batch trong tên (dạng L25_V001 mới hợp lệ)')
if redundant:
    print(f'[WARN] bỏ qua {redundant} file trùng-lặp (cùng tên & cùng dung lượng ở '
          'folder khác, vd Videos_L25_a vs Videos_L25_a1) -> giữ bản gặp đầu tiên.')
if conflicts:
    print(f'[LỖI] {len(conflicts)} tên video trùng NHƯNG khác nội dung (khác dung lượng):')
    for name, a, b in conflicts[:20]:
        print(f'  {name}: {a}  <->  {b}')
    raise SystemExit(
        f'{len(conflicts)} video khác nội dung dùng chung tên -> video_id không '
        'duy nhất trong batch. Cần đưa hậu tố folder (_a/_b) vào khóa và sửa đồng '
        'bộ các bước sau. Xem danh sách ở trên.'
    )

In [ ]:
def sample_frames_from_shot(start_idx, end_idx):
    """First / middle / last frame of a shot (matches the AIC-HCMC-2025 heuristic)."""
    mid = start_idx + (end_idx - start_idx) // 2
    return sorted({start_idx, mid, end_idx})


def frame_signature(frame):
    """Chữ ký nhẹ để so gần-trùng: ảnh xám 32x32, trừ trung bình, L2-normalize.
    Dot của 2 chữ ký = tương quan (cosine) trong [-1, 1]. Trừ trung bình giúp
    ít nhạy với thay đổi độ sáng tổng thể hơn là cosine trên pixel thô."""
    g = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    g = cv2.resize(g, (32, 32), interpolation=cv2.INTER_AREA).astype('float32').reshape(-1)
    g -= g.mean()
    return g / (float(g @ g) ** 0.5 + 1e-9)


def save_webp(frame_bgr, out_path):
    """Ghi thẳng frame BGR (OpenCV) ra webp đã thu nhỏ - gộp việc của Step 3 cũ.

    Ghi ra file .tmp rồi os.replace: đầy đĩa giữa chừng sẽ KHÔNG để lại file webp
    cụt mà lần chạy sau tưởng là đã xong."""
    img = Image.fromarray(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
    if REDUCE_RATIO != 1.0:
        x, y = img.size
        img = img.resize((max(1, math.floor(x * REDUCE_RATIO)),
                          max(1, math.floor(y * REDUCE_RATIO))), Image.LANCZOS)
    tmp = f'{out_path}.tmp'
    img.save(tmp, format='webp', optimize=True, quality=WEBP_QUALITY)   # OSError nếu hết đĩa
    os.replace(tmp, out_path)


def load_json(path, default):
    if not os.path.exists(path):
        return default
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def save_json_atomic(obj, path):
    """Ghi tmp rồi replace: crash / đầy đĩa cũng không làm hỏng file cũ.

    Không dùng indent - frames_meta có thể tới hàng trăm nghìn entry, indent làm
    file phình vài lần mà backend chỉ json.loads nên không cần đẹp."""
    tmp = f'{path}.tmp'
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False)
    os.replace(tmp, path)


def free_gb(path):
    return shutil.disk_usage(path).free / (1024 ** 3)

In [ ]:
frames_meta = load_json(META_PATH, {})
done = set(load_json(DONE_PATH, []))
print(f'đã có {len(frames_meta)} frame trong meta, {len(done)} video đã xong -> sẽ bỏ qua')

todo_videos = [(b, v) for b, vs in all_video_paths.items() for v in sorted(vs)
               if f'{b}_{v}' not in done]
print(f'còn {len(todo_videos)} video cần cắt, đĩa trống {free_gb(SAVE_DIR):.1f} GB')

In [ ]:
kept = dropped = 0
stopped_reason = None

for batch, videos in all_video_paths.items():
    pending = [v for v in sorted(videos) if f'{batch}_{v}' not in done]
    if not pending:
        continue
    for n, video_id in enumerate(tqdm(pending, desc=batch), 1):
        video_name = f'{batch}_{video_id}'                 # flat: L21_V001

        # Hết đĩa thì dừng CÓ KIỂM SOÁT, trước khi ghi hỏng thứ gì. Bản cũ để
        # cv2.imwrite trả False âm thầm -> frame biến mất không dấu vết, còn
        # papermill thì chết với traceback JSON khó hiểu.
        if free_gb(SAVE_DIR) < MIN_FREE_GB:
            stopped_reason = (f'hết đĩa ở {video_name} (còn {free_gb(SAVE_DIR):.2f} GB '
                              f'< MIN_FREE_GB={MIN_FREE_GB}). Đã lưu tiến độ; hãy lưu '
                              'output thành dataset rồi chạy tiếp - sẽ resume từ đây.')
            break

        scene_path = f'{SCENE_JSON_DIR}/{batch}/{video_id}.json'
        if not os.path.exists(scene_path):
            print('missing scenes:', scene_path)
            continue
        with open(scene_path, 'r') as f:
            scenes = json.load(f)

        out_dir = f'{SAVE_DIR}/{batch}/{video_name}'      # nested: Keyframes/L21/L21_V001
        os.makedirs(out_dir, exist_ok=True)

        # All frame indices we want for this video (first/mid/last of each shot)
        wanted = sorted({
            idx
            for shot in scenes
            for idx in sample_frames_from_shot(int(shot[0]), int(shot[1]))
        })

        cap = cv2.VideoCapture(videos[video_id])
        if not cap.isOpened():
            print('[WARN] không mở được video:', videos[video_id])
            cap.release()
            continue
        fps = cap.get(cv2.CAP_PROP_FPS) or 25.0

        # Sequential decode: read frames in order and pick the wanted ones.
        # Avoids H.264 random seeking (cap.set(POS_FRAMES, ...)), which is not
        # frame-accurate and triggers the "mmco: unref short failure" warnings /
        # decode artifacts when landing mid-GOP without reference frames.
        todo = set(wanted)
        max_idx = wanted[-1] if wanted else -1
        idx = 0
        last_sig = None          # reset dedup theo từng video
        while todo and idx <= max_idx:
            ret, frame = cap.read()
            if not ret:
                break
            if idx in todo:
                todo.discard(idx)
                # Bỏ frame gần-trùng với frame giữ gần nhất trong video này
                if DEDUP_THRESHOLD > 0:
                    sig = frame_signature(frame)
                    if last_sig is not None and float(sig @ last_sig) > DEDUP_THRESHOLD:
                        dropped += 1
                        idx += 1
                        continue
                    last_sig = sig
                fname = f'{idx:06d}.{FINAL_EXT}'
                fpath = f'{out_dir}/{fname}'
                if not os.path.exists(fpath):
                    save_webp(frame, fpath)              # ném OSError nếu hết đĩa
                kept += 1
                frames_meta[f'{video_name}_{idx:06d}'] = {
                    'video_name': video_name,
                    'frame_name': fname,
                    'timestamp_ms': int(idx / fps * 1000),
                    'fps': float(fps),
                }
            idx += 1
        cap.release()

        done.add(video_name)
        if n % FLUSH_EVERY == 0:                          # ghi tiến độ dần -> resume được
            save_json_atomic(frames_meta, META_PATH)
            save_json_atomic(sorted(done), DONE_PATH)

    save_json_atomic(frames_meta, META_PATH)
    save_json_atomic(sorted(done), DONE_PATH)
    if stopped_reason:
        break

save_json_atomic(frames_meta, META_PATH)
save_json_atomic(sorted(done), DONE_PATH)

print(f'frames kept: {kept}, dropped near-dup: {dropped} (cosine > {DEDUP_THRESHOLD})')
print(f'meta: {len(frames_meta)} frame -> {META_PATH}')
print(f'video xong: {len(done)}, đĩa trống còn {free_gb(SAVE_DIR):.1f} GB')
if stopped_reason:
    raise SystemExit('[DỪNG] ' + stopped_reason)

### Kiểm tra trước khi export dataset

Đếm file webp thực tế trên đĩa và đối chiếu với `frames_meta.json`. Hai số phải
bằng nhau - lệch nghĩa là có frame ghi rồi mà thiếu metadata (hoặc ngược lại), và
các bước sau (FAISS id_map, OCR) sẽ join hụt.

In [ ]:
sizes = [p.stat().st_size for p in Path(SAVE_DIR).rglob(f'*.{FINAL_EXT}')]
n_disk, size_gb = len(sizes), sum(sizes) / (1024 ** 3)
print(f'webp trên đĩa   : {n_disk}')
print(f'entry trong meta: {len(frames_meta)}')
print(f'dung lượng      : {size_gb:.2f} GB')
if n_disk != len(frames_meta):
    print('[WARN] lệch số lượng - kiểm tra lại trước khi chạy Step 4')

leftovers = list(Path(SAVE_DIR).rglob('*.tmp'))
if leftovers:
    print(f'[WARN] còn {len(leftovers)} file .tmp (ghi dở) - xóa đi:')
    for p in leftovers[:5]:
        print(' ', p)